In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/demand_forecasting.csv")

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [4]:
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [6]:
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

In [7]:
df['Price_Difference'] = (
    df['Price'] - df['Competitor Pricing']
)

In [8]:
df['Price_Ratio'] = (
    df['Price'] / df['Competitor Pricing']
)

In [9]:
df['Effective_Price'] = (
    df['Price'] * (1 - df['Discount'] / 100)
)

In [10]:
df['Promotion_Discount'] = (
    df['Promotion'] * df['Discount']
)

In [11]:
df['Low_Inventory'] = (
    df['Inventory Level'] < df['Inventory Level'].median()
).astype(int)

In [12]:
df.to_csv(
    "../data/demand_forecasting_engineered.csv",
    index=False
)

In [13]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,76000,2023-01-15 12:00:00,2022-01-01 00:00:00,2022-07-09 18:00:00,2023-01-15 12:00:00,2023-07-24 06:00:00,2024-01-30 00:00:00,NaN
Inventory Level,76000.0,301.062842,0.0,136.0,227.0,408.0,2267.0,226.510161
Units Sold,76000.0,88.827316,0.0,58.0,84.0,114.0,426.0,43.994525
Units Ordered,76000.0,89.090645,0.0,0.0,0.0,121.0,1616.0,162.404627
Price,76000.0,67.726028,4.74,31.9975,64.5,95.83,228.03,39.377899
Discount,76000.0,9.087039,0.0,5.0,10.0,10.0,25.0,7.475781
Promotion,76000.0,0.328947,0.0,0.0,0.0,1.0,1.0,0.469834
Competitor Pricing,76000.0,69.454029,4.29,32.62,65.7,97.9325,261.22,40.943818
Epidemic,76000.0,0.2,0.0,0.0,0.0,0.0,1.0,0.400003
Demand,76000.0,104.317158,4.0,71.0,100.0,133.0,430.0,46.964801


In [14]:
feature_cols = [
    'Price',
    'Discount',
    'Competitor Pricing',
    'Price_Difference',
    'Price_Ratio',
    'Effective_Price',
    'Promotion',
    'Promotion_Discount',
    'Inventory Level',
    'Low_Inventory',
    'Demand'
]

df[feature_cols].corr()['Demand'].sort_values(ascending=False)

Demand                1.000000
Promotion             0.282537
Promotion_Discount    0.264054
Discount              0.224723
Inventory Level       0.126618
Price_Difference      0.002196
Price_Ratio           0.001611
Competitor Pricing   -0.023036
Price                -0.023461
Effective_Price      -0.054218
Low_Inventory        -0.085850
Name: Demand, dtype: float64

In [15]:
df = df.sort_values(['Store ID', 'Product ID', 'Date'])

In [16]:
df['Previous_Price'] = (
    df.groupby(['Store ID', 'Product ID'])['Price']
      .shift(1)
)

In [17]:
df['Price_Change'] = (
    df['Price'] - df['Previous_Price']
)

In [18]:
df['Price_Change_Pct'] = (
    df['Price_Change'] / df['Previous_Price']
) * 100

In [19]:
df[
    [
        'Store ID',
        'Product ID',
        'Date',
        'Price',
        'Previous_Price',
        'Price_Change',
        'Price_Change_Pct'
    ]
].head(20)

,Store ID,Product ID,Date,Price,Previous_Price,Price_Change,Price_Change_Pct
0,S001,P0001,2022-01-01,72.72,NaN,NaN,NaN
100,S001,P0001,2022-01-02,65.63,72.72,-7.09,-9.749725
200,S001,P0001,2022-01-03,68.55,65.63,2.92,4.449185
300,S001,P0001,2022-01-04,61.66,68.55,-6.89,-10.051058
400,S001,P0001,2022-01-05,59.56,61.66,-2.10,-3.405774
500,S001,P0001,2022-01-06,84.03,59.56,24.47,41.084621
600,S001,P0001,2022-01-07,66.07,84.03,-17.96,-21.373319
700,S001,P0001,2022-01-08,68.56,66.07,2.49,3.768730
800,S001,P0001,2022-01-09,62.76,68.56,-5.80,-8.459743
900,S001,P0001,2022-01-10,68.99,62.76,6.23,9.926705


In [20]:
df[['Previous_Price', 'Price_Change', 'Price_Change_Pct']].isnull().sum()

Previous_Price      100
Price_Change        100
Price_Change_Pct    100
dtype: int64

In [21]:
df[['Previous_Price', 'Price_Change', 'Price_Change_Pct']].describe()

,Previous_Price,Price_Change,Price_Change_Pct
count,75900.000000,75900.000000,75900.000000
mean,67.722841,0.001944,1.658918
std,39.375483,13.283852,18.611985
min,4.740000,-79.850000,-57.500310
25%,31.990000,-6.230000,-11.119773
50%,64.500000,0.020000,0.034830
75%,95.830000,6.220000,12.586436
max,228.030000,81.710000,120.460199


In [22]:
df['Previous_Demand'] = (
    df.groupby(['Store ID', 'Product ID'])['Demand']
      .shift(1)
)

In [23]:
df[
    [
        'Store ID',
        'Product ID',
        'Date',
        'Demand',
        'Previous_Demand'
    ]
].head(20)

,Store ID,Product ID,Date,Demand,Previous_Demand
0,S001,P0001,2022-01-01,115,NaN
100,S001,P0001,2022-01-02,84,115.0
200,S001,P0001,2022-01-03,132,84.0
300,S001,P0001,2022-01-04,67,132.0
400,S001,P0001,2022-01-05,110,67.0
500,S001,P0001,2022-01-06,146,110.0
600,S001,P0001,2022-01-07,87,146.0
700,S001,P0001,2022-01-08,113,87.0
800,S001,P0001,2022-01-09,87,113.0
900,S001,P0001,2022-01-10,99,87.0


In [24]:
df['Previous_7Day_Avg_Demand'] = (
    df.groupby(['Store ID', 'Product ID'])['Demand']
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [25]:
df[
    [
        'Store ID',
        'Product ID',
        'Date',
        'Demand',
        'Previous_Demand',
        'Previous_7Day_Avg_Demand'
    ]
].head(15)

,Store ID,Product ID,Date,Demand,Previous_Demand,Previous_7Day_Avg_Demand
0,S001,P0001,2022-01-01,115,NaN,NaN
100,S001,P0001,2022-01-02,84,115.0,NaN
200,S001,P0001,2022-01-03,132,84.0,NaN
300,S001,P0001,2022-01-04,67,132.0,NaN
400,S001,P0001,2022-01-05,110,67.0,NaN
500,S001,P0001,2022-01-06,146,110.0,NaN
600,S001,P0001,2022-01-07,87,146.0,NaN
700,S001,P0001,2022-01-08,113,87.0,105.857143
800,S001,P0001,2022-01-09,87,113.0,105.571429
900,S001,P0001,2022-01-10,99,87.0,106.000000


In [26]:
df['Demand_Trend'] = (
    df['Previous_Demand'] - df['Previous_7Day_Avg_Demand']
)

In [27]:
df[
    [
        'Date',
        'Demand',
        'Previous_Demand',
        'Previous_7Day_Avg_Demand',
        'Demand_Trend'
    ]
].head(15)

,Date,Demand,Previous_Demand,Previous_7Day_Avg_Demand,Demand_Trend
0,2022-01-01,115,NaN,NaN,NaN
100,2022-01-02,84,115.0,NaN,NaN
200,2022-01-03,132,84.0,NaN,NaN
300,2022-01-04,67,132.0,NaN,NaN
400,2022-01-05,110,67.0,NaN,NaN
500,2022-01-06,146,110.0,NaN,NaN
600,2022-01-07,87,146.0,NaN,NaN
700,2022-01-08,113,87.0,105.857143,-18.857143
800,2022-01-09,87,113.0,105.571429,7.428571
900,2022-01-10,99,87.0,106.000000,-19.000000


In [28]:
df['Inventory_Coverage_Days'] = (
    df['Inventory Level'] /
    df['Previous_7Day_Avg_Demand']
)

In [29]:
df[
    [
        'Date',
        'Inventory Level',
        'Previous_7Day_Avg_Demand',
        'Inventory_Coverage_Days'
    ]
].head(15)

,Date,Inventory Level,Previous_7Day_Avg_Demand,Inventory_Coverage_Days
0,2022-01-01,195,NaN,NaN
100,2022-01-02,93,NaN,NaN
200,2022-01-03,274,NaN,NaN
300,2022-01-04,132,NaN,NaN
400,2022-01-05,319,NaN,NaN
500,2022-01-06,190,NaN,NaN
600,2022-01-07,92,NaN,NaN
700,2022-01-08,308,105.857143,2.909582
800,2022-01-09,212,105.571429,2.008119
900,2022-01-10,403,106.000000,3.801887


In [30]:
# Interaction Features

df['Weekend_Promotion'] = (
    df['IsWeekend'] * df['Promotion']
)

In [31]:
df[
    [
        'Date',
        'IsWeekend',
        'Promotion',
        'Weekend_Promotion'
    ]
].head(20)

,Date,IsWeekend,Promotion,Weekend_Promotion
0,2022-01-01,1,0,0
100,2022-01-02,1,0,0
200,2022-01-03,0,1,0
300,2022-01-04,0,0,0
400,2022-01-05,0,1,0
500,2022-01-06,0,0,0
600,2022-01-07,0,0,0
700,2022-01-08,1,1,1
800,2022-01-09,1,0,0
900,2022-01-10,0,0,0


In [ ]:
# Year
# Month
# Day
# DayOfWeek
# IsWeekend


# Price
# Discount
# Price_Difference
# Price_Ratio
# Effective_Price
# Promotion_Discount
# Price_Change
# Price_Change_Pct



# Previous_Demand
# Previous_7Day_Avg_Demand
# Demand_Trend


# Low_Inventory
# Inventory_Coverage_Days

# Weekend_Promotion



In [32]:
df.isnull().sum().sort_values(ascending=False)

Demand_Trend                700
Previous_7Day_Avg_Demand    700
Inventory_Coverage_Days     700
Previous_Demand             100
Price_Change                100
Previous_Price              100
Price_Change_Pct            100
Date                          0
Store ID                      0
Product ID                    0
Weather Condition             0
Category                      0
Region                        0
Inventory Level               0
Units Sold                    0
Units Ordered                 0
Price                         0
Discount                      0
Month                         0
Year                          0
Demand                        0
Epidemic                      0
Seasonality                   0
Competitor Pricing            0
Promotion                     0
Day                           0
Low_Inventory                 0
Promotion_Discount            0
Effective_Price               0
Price_Ratio                   0
Price_Difference              0
IsWeeken

In [33]:
df = df.dropna(
    subset=['Previous_7Day_Avg_Demand']
).copy()

In [34]:
df.shape

(75300, 34)

In [35]:
df.isnull().sum().sort_values(ascending=False).head(10)

Date               0
Store ID           0
Product ID         0
Category           0
Region             0
Inventory Level    0
Units Sold         0
Units Ordered      0
Price              0
Discount           0
dtype: int64

In [36]:
df.to_csv(
    "../data/demand_forecasting_engineered.csv",
    index=False
)